# Full silicon and oxygen XPS analysis

**Instructor solution.** Run this notebook independently from top to bottom.

In [1]:
from pathlib import Path
import sys

# Support kernels started in the repository root or in instructor-notebooks.
ROOT = Path.cwd()
if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data").is_dir():
    raise FileNotFoundError("Open this notebook from the repository or instructor-notebooks folder.")
sys.path.insert(0, str(ROOT))

Rebuild both spectra, subtract example linear backgrounds, fit the silicon and oxygen features, and compare integrated areas.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px

from scipy.signal import savgol_filter, find_peaks
from scipy.optimize import curve_fit
from scipy.integrate import simpson, quad

In [3]:
# data loading
si_peaks = pd.read_table(ROOT / "si_data/N0125702.asc", header=None)

# add column names
si_peaks.columns = ["energy", "intensity"]

# visualize
si_fig = px.line(si_peaks, x="energy", y="intensity")
si_fig.update_xaxes(autorange="reversed")
si_fig.show()

In [4]:
# define what we want to fit for background subtraction
background_index = (98.5 > si_peaks["energy"]) | (si_peaks["energy"] > 106.5)
background = si_peaks[background_index]
background.head()

,energy,intensity
0,112.000,3262.6982
1,111.875,3229.2760
2,111.750,3138.2208
3,111.625,3037.6669
4,111.500,3043.9405


In [5]:
# Fit a polynomial of degree 1
fit = np.polyfit(background["energy"], background["intensity"], 1)

print(fit)

[   137.77256437 -12432.6279266 ]


In [6]:
# Get fit values
vals = np.polyval(fit, si_peaks["energy"])

In [7]:
# Subtract the background
si_peaks["corrected_intensity"] = si_peaks["intensity"] - vals

In [8]:
def gaussian(x, A, B, C):
    y = A*np.exp(-(x-B)**2/(2*C**2)) 
    return y

def double_gaussian(x, a1, b1, c1, a2, b2, c2):
    g1 = a1 * np.exp(-(x - b1)**2 / (2 * c1**2))
    g2 = a2 * np.exp(-(x - b2)**2 / (2 * c2**2))
    return g1 + g2

In [9]:
# Pick a peak 
peak_1 = si_peaks[si_peaks["energy"].between(99.25, 101.5)]
peak_1.head()

,energy,intensity,corrected_intensity
84,101.500,5369.8225,3818.535143
85,101.375,6495.5969,4961.531114
86,101.250,8029.8160,6512.971784
87,101.125,10087.1710,8587.548355
88,101.000,12690.4160,11208.014925


In [10]:
parameters_1, covariance_1 = curve_fit(
    double_gaussian,
    peak_1["energy"],
    peak_1["corrected_intensity"],
    p0=[30000, 100.0, 0.1, 15000, 100.5, 0.1],
    bounds=(
        [0, 99.25, 0.01, 0, 99.25, 0.01],
        [np.inf, 101.5, 1, np.inf, 101.5, 1],
    ),
)


In [11]:
o_peaks = pd.read_table(ROOT / "si_data/N0125704.asc", header=None, names=["energy", "intensity"])
o_peaks.head()

,energy,intensity
0,544.000,8847.9422
1,543.875,8855.1659
2,543.750,8822.7821
3,543.625,8804.8229
4,543.500,8678.1587


In [12]:
o_fig = px.line(o_peaks, x="energy", y="intensity", title="Oxygen spectrum and selected background")
o_fig.update_xaxes(autorange="reversed", title="Binding energy (eV)")
o_fig.update_yaxes(title="Intensity (source units)")
o_fig.show()

In [13]:
o_background = o_peaks[(o_peaks["energy"] < 529) | (o_peaks["energy"] > 538)]
o_background.head()

,energy,intensity
0,544.000,8847.9422
1,543.875,8855.1659
2,543.750,8822.7821
3,543.625,8804.8229
4,543.500,8678.1587


In [14]:
o_background_fit = np.polyfit(o_background["energy"], o_background["intensity"], 1)
o_baseline = np.polyval(o_background_fit, o_peaks["energy"])

In [15]:
o_fig.add_scatter(x=o_background["energy"], y=o_background["intensity"], mode="markers", name="Background points")
o_fig.add_scatter(x=o_peaks["energy"], y=o_baseline, mode="lines", line_dash="dash", name="Linear background")
o_fig.show()

In [16]:
o_peaks["corrected_intensity"] = o_peaks["intensity"] - o_baseline
o_corrected_fig = px.line(o_peaks, x="energy", y="corrected_intensity", title="Oxygen spectrum after background subtraction", labels={"energy": "Binding energy (eV)", "corrected_intensity": "Corrected intensity (source units)"})
o_corrected_fig.update_xaxes(autorange="reversed")
o_corrected_fig.show()

In [17]:
si_peaks = si_peaks.sort_values("energy")
si_peak_area = simpson(y=si_peaks["corrected_intensity"], x=si_peaks["energy"])

In [18]:
o_peaks = o_peaks.sort_values("energy")
o_peak_area = simpson(y=o_peaks["corrected_intensity"], x=o_peaks["energy"])
print("O integrated area:", o_peak_area)

O integrated area: 135583.24309687433


In [19]:
si_to_o_area_ratio = si_peak_area / o_peak_area
print("Si/O raw area ratio:", si_to_o_area_ratio)

Si/O raw area ratio: 0.5218457045907591


In [20]:
pd.DataFrame({"Spectrum": ["Si", "O"], "Numerical area": [si_peak_area, o_peak_area]})

,Spectrum,Numerical area
0,Si,70753.533025
1,O,135583.243097


In [21]:
peak_2 = si_peaks[si_peaks["energy"].between(102, 106.5)]
parameters_2, covariance_2 = curve_fit(gaussian, peak_2["energy"], peak_2["corrected_intensity"], p0=[peak_2["corrected_intensity"].max(), 104, 0.7], bounds=([0, 102, 0.01], [np.inf, 106.5, 3]))
print("Second Si peak parameters:", parameters_2)

Second Si peak parameters: [1.25415441e+04 1.04134654e+02 7.92619998e-01]


In [22]:
o_fit_region = o_peaks[o_peaks["energy"].between(529, 538)]
parameters_o, covariance_o = curve_fit(gaussian, o_fit_region["energy"], o_fit_region["corrected_intensity"], p0=[o_fit_region["corrected_intensity"].max(), 533.4, 0.7], bounds=([0, 529, 0.01], [np.inf, 538, 3]))
print("O peak parameters:", parameters_o)

O peak parameters: [7.18267350e+04 5.33456578e+02 7.32167038e-01]


In [23]:
fig_si2 = px.line(peak_2, x="energy", y="corrected_intensity", title="Second silicon peak and Gaussian fit")
fig_si2.add_scatter(x=peak_2["energy"], y=gaussian(peak_2["energy"], *parameters_2), mode="lines", line_dash="dash", name="Gaussian model")
fig_si2.update_xaxes(autorange="reversed", title="Binding energy (eV)")
fig_si2.update_yaxes(title="Corrected intensity (source units)")
fig_si2.show()

In [24]:
fig_o_fit = px.line(o_fit_region, x="energy", y="corrected_intensity", title="Oxygen peak and Gaussian fit")
fig_o_fit.add_scatter(x=o_fit_region["energy"], y=gaussian(o_fit_region["energy"], *parameters_o), mode="lines", line_dash="dash", name="Gaussian model")
fig_o_fit.update_xaxes(autorange="reversed", title="Binding energy (eV)")
fig_o_fit.update_yaxes(title="Corrected intensity (source units)")
fig_o_fit.show()

In [25]:
si_area_1, si_error_1 = quad(double_gaussian, 98, 102, args=tuple(parameters_1))
si_area_2, si_error_2 = quad(gaussian, 102, 108, args=tuple(parameters_2))
o_fit_area, o_fit_error = quad(gaussian, 529, 538, args=tuple(parameters_o))
print("Fitted areas (Si first, Si second, O):", si_area_1, si_area_2, o_fit_area)

Fitted areas (Si first, Si second, O): 44876.80986350075 24829.39183296293 131821.49477068605


In [26]:
fitted_area_ratio = (si_area_1 + si_area_2) / o_fit_area
print("Si/O fitted area ratio:", fitted_area_ratio)
print("Si/O numerical area ratio:", si_to_o_area_ratio)

Si/O fitted area ratio: 0.528792378039129
Si/O numerical area ratio: 0.5218457045907591


In [27]:
pd.DataFrame({"Peak": ["Si first", "Si second", "O"], "Fitted area": [si_area_1, si_area_2, o_fit_area], "Quadrature error": [si_error_1, si_error_2, o_fit_error]})

,Peak,Fitted area,Quadrature error
0,Si first,44876.809864,9.711383e-05
1,Si second,24829.391833,7.188607e-08
2,O,131821.494771,1.486442e-09


**Instructor interpretation:** Use oxygen energies below 529 eV and above 538 eV as one plausible background selection. Inspect the overlaid points and fitted line before subtraction; different endpoint choices change the integrated areas. The corrected oxygen peak is near 533.4 eV. A single Gaussian is a teaching approximation; inspect deviations between model and data. The first silicon feature uses the lesson's double Gaussian and the second uses a single Gaussian.

The printed Si/O values are **raw intensity-area ratios**. Atomic ratios require appropriate relative sensitivity factors and compatible acquisition settings; the silicon substrate and oxide also contribute differently. Do not infer SiO₂ stoichiometry directly from these ratios. Numerical integration and fitted integration use different windows and describe the tails differently. Quadrature error estimates numerical integration error only, excluding fit and background uncertainty.

References: [Thermo Fisher oxygen XPS reference](https://www.thermofisher.cn/hk/zt/home/materials-science/learning-center/periodic-table/non-metal/oxygen.html); [Avoiding common errors in XPS analysis](https://doi.org/10.1016/j.apsadv.2023.100534).

The tables below provide the measured and corrected values underlying the spectra.

In [28]:
display(si_peaks)
display(o_peaks)

,energy,intensity,corrected_intensity
160,92.000,634.7571,392.309105
159,92.125,627.9534,368.283834
158,92.250,608.5805,331.689364
157,92.375,599.8107,305.697993
156,92.500,600.6060,289.271723
...,...,...,...
4,111.500,3043.9405,114.927500
3,111.625,3037.6669,91.432329
2,111.750,3138.2208,174.764658
1,111.875,3229.2760,248.598288


,energy,intensity,corrected_intensity
176,522.000,7399.8304,398.093237
175,522.125,7233.5145,223.967493
174,522.250,7240.2299,222.873048
173,522.375,7296.9571,271.790404
172,522.500,7277.5399,244.563359
...,...,...,...
4,543.500,8678.1587,333.128269
3,543.625,8804.8229,451.982625
2,543.750,8822.7821,462.131980
1,543.875,8855.1659,486.705936
